In [1]:
import sys
print(sys.executable)

c:\Users\TUSHAR\Desktop\sentiment project\.venv\Scripts\python.exe


In [3]:
# Import Dependancy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
df = pd.read_csv("sentiment data/train.csv",encoding='latin-1')
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [14]:
df.shape


(27481, 10)

In [15]:
df=df[['text','sentiment']]

In [16]:
df.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [17]:
df.sentiment.value_counts()

sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64

In [18]:
df.isnull().sum()

text         1
sentiment    0
dtype: int64

In [19]:
df=df.dropna(subset=['text'])

In [20]:
print(df.isnull().sum())
print(df.shape)

text         0
sentiment    0
dtype: int64
(27480, 2)


In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
df['text'].head(10)

0                  I`d have responded, if I were going
1        Sooo SAD I will miss you here in San Diego!!!
2                            my boss is bullying me...
3                       what interview! leave me alone
4     Sons of ****, why couldn`t they put them on t...
5    http://www.dothebouncy.com/smf - some shameles...
6    2am feedings for the baby are fun when he is a...
7                                           Soooo high
8                                          Both of you
9     Journey!? Wow... u just became cooler.  hehe....
Name: text, dtype: str

## cleaning the data

In [25]:
import re
import emoji

def clean_text(text):
    text = str(text)
    
    # Convert emojis to text
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [26]:
df[['text', 'clean_text']].head(10)

,text,clean_text
0,"I`d have responded, if I were going","i`d have responded, if i were going"
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad i will miss you here in san diego!!!
2,my boss is bullying me...,my boss is bullying me...
3,what interview! leave me alone,what interview! leave me alone
4,"Sons of ****, why couldn`t they put them on t...","sons of ****, why couldn`t they put them on th..."
5,http://www.dothebouncy.com/smf - some shameles...,- some shameless plugging for the best rangers...
6,2am feedings for the baby are fun when he is a...,2am feedings for the baby are fun when he is a...
7,Soooo high,soooo high
8,Both of you,both of you
9,Journey!? Wow... u just became cooler. hehe....,journey!? wow... u just became cooler. hehe......


In [27]:
df['sentiment'] = df['sentiment'].map({
    'negative': 0,
    'neutral': 1,
    'positive': 2
})

df['sentiment'].value_counts()

sentiment
1    11117
2     8582
0     7781
Name: count, dtype: int64

## **train test split**